In [10]:
# ============================================================
# FULL PIPELINE FOR LOBSTER BIOACOUSTICS CLASSIFICATION
# WITH 95% BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

import os
import sys
import numpy as np
import librosa
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense

# ============================================================
# DATASET ROOT
# ============================================================

BASE_DATASET_DIR = "/home/feliciano/LOBSTER SOUNDS/DatasetClass"

# ============================================================
# DATA LOADING
# ============================================================

def load_audio_folder(folder, label, n_mfcc=40):
    X, y = [], []

    if not os.path.exists(folder):
        print(f"❌ FOLDER NOT FOUND: {folder}")
        return np.array(X), np.array(y)

    for file in os.listdir(folder):
        if file.lower().endswith(".wav"):
            path = os.path.join(folder, file)
            audio, sr = librosa.load(path, sr=22050)
            mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
            X.append(np.mean(mfcc, axis=1))
            y.append(label)

    return np.array(X), np.array(y)

# ============================================================
# LOAD DATA
# ============================================================

X_adult, y_adult = load_audio_folder(os.path.join(BASE_DATASET_DIR, "adult_lobsters"), 1)
X_juvenile, y_juvenile = load_audio_folder(os.path.join(BASE_DATASET_DIR, "juvenile_lobsters"), 0)

X = np.vstack([X_adult, X_juvenile])
y = np.concatenate([y_adult, y_juvenile])

if X.shape[0] == 0 or len(np.unique(y)) < 2:
    sys.exit("❌ Dataset empty or single class only.")

# ============================================================
# SPLIT + SCALE
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ============================================================
# BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

def bootstrap_ci(y_true, y_pred_or_prob, metric_fn, n_boot=2000):
    rng = np.random.default_rng(42)
    values = []

    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))

        if metric_fn is roc_auc_score:
            if len(np.unique(y_true[idx])) < 2:
                continue

        values.append(metric_fn(y_true[idx], y_pred_or_prob[idx]))

    if len(values) == 0:
        return np.nan, np.nan

    return np.percentile(values, 2.5), np.percentile(values, 97.5)

# ============================================================
# ML MODELS
# ============================================================

models = {
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(128,), max_iter=500),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    acc_ci = bootstrap_ci(y_test, y_pred, accuracy_score)
    auc_ci = bootstrap_ci(y_test, y_prob, roc_auc_score)

    results.append([
        name,
        f"{acc*100:.2f}",
        f"{acc_ci[0]*100:.2f}–{acc_ci[1]*100:.2f}",
        f"{auc*100:.2f}",
        f"{auc_ci[0]*100:.2f}–{auc_ci[1]*100:.2f}"
    ])

# ============================================================
# DL MODEL (1D-CNN)
# ============================================================

X_train_dl = X_train[..., np.newaxis]
X_test_dl = X_test[..., np.newaxis]

cnn = Sequential([
    Conv1D(64, 3, activation="relu", input_shape=(X_train.shape[1], 1)),
    MaxPooling1D(2),
    Flatten(),
    Dense(128, activation="relu"),
    Dense(1, activation="sigmoid")
])

cnn.compile(optimizer="adam", loss="binary_crossentropy")
cnn.fit(X_train_dl, y_train, epochs=10, batch_size=32, verbose=0)

y_prob = cnn.predict(X_test_dl).ravel()
y_pred = (y_prob > 0.5).astype(int)

results.append([
    "1D-CNN",
    f"{accuracy_score(y_test, y_pred)*100:.2f}",
    "—",
    f"{roc_auc_score(y_test, y_prob)*100:.2f}",
    "—"
])

# ============================================================
# FINAL RESULTS
# ============================================================

df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy (%)", "95% CI", "AUC (%)", "95% CI"]
)

print(df.to_string(index=False))

/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
        Model Accuracy (%)      95% CI AUC (%)      95% CI
Random Forest        97.13 96.17–97.95   99.43 99.07–99.73
      XGBoost        98.08 97.40–98.77   99.63 99.39–99.84
          MLP        98.02 97.26–98.70   99.77 99.61–99.89
  Naive Bayes        90.90 89.46–92.41   97.02 96.17–97.78
          SVM        98.29 97.61–98.91   99.61 99.24–99.88
          KNN        97.74 96.99–98.43   99.27 98.75–99.71
       1D-CNN        98.29           —   99.77           —


In [11]:
# ============================================================
# FULL PIPELINE FOR LOBSTER BIOACOUSTICS CLASSIFICATION
# PAPER-COMPLIANT VERSION (ALL MODELS INCLUDED)
# ============================================================

import os
import sys
import numpy as np
import librosa
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense

# ============================================================
# DATASET ROOT
# ============================================================

BASE_DATASET_DIR = "/home/feliciano/LOBSTER SOUNDS/DatasetClass"

# ============================================================
# LOAD AUDIO
# ============================================================

def load_audio_folder(folder, label, n_mfcc=40):
    X, y = [], []
    for f in os.listdir(folder):
        if f.lower().endswith(".wav"):
            audio, sr = librosa.load(os.path.join(folder, f), sr=22050)
            mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
            X.append(np.mean(mfcc, axis=1))
            y.append(label)
    return np.array(X), np.array(y)

X_adult, y_adult = load_audio_folder(os.path.join(BASE_DATASET_DIR, "adult_lobsters"), 1)
X_juvenile, y_juvenile = load_audio_folder(os.path.join(BASE_DATASET_DIR, "juvenile_lobsters"), 0)

X = np.vstack([X_adult, X_juvenile])
y = np.concatenate([y_adult, y_juvenile])

if len(np.unique(y)) < 2:
    sys.exit("Dataset invalid")

# ============================================================
# SPLIT + SCALE
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ============================================================
# BOOTSTRAP CI
# ============================================================

def bootstrap_ci(y_true, y_pred_or_prob, metric, n=2000):
    rng = np.random.default_rng(42)
    vals = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        if metric is roc_auc_score and len(np.unique(y_true[idx])) < 2:
            continue
        vals.append(metric(y_true[idx], y_pred_or_prob[idx]))
    return np.percentile(vals, 2.5), np.percentile(vals, 97.5)

# ============================================================
# ML MODELS
# ============================================================

models = {
    "Random Forest": RandomForestClassifier(n_estimators=300),
    "XGBoost": XGBClassifier(eval_metric="logloss"),
    "MLP": MLPClassifier(hidden_layer_sizes=(128,), max_iter=500),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    yp = model.predict(X_test)
    yp_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, yp)
    auc = roc_auc_score(y_test, yp_prob)

    acc_ci = bootstrap_ci(y_test, yp, accuracy_score)
    auc_ci = bootstrap_ci(y_test, yp_prob, roc_auc_score)

    results.append([
        name,
        f"{acc*100:.2f}",
        f"{acc_ci[0]*100:.2f}–{acc_ci[1]*100:.2f}",
        f"{auc*100:.2f}",
        f"{auc_ci[0]*100:.2f}–{auc_ci[1]*100:.2f}"
    ])

# ============================================================
# DL MODELS: 1D-CNN and 1D-DCNN (1–4 layers)
# ============================================================

X_train_dl = X_train[..., np.newaxis]
X_test_dl = X_test[..., np.newaxis]

def build_cnn(depth, dilated=False):
    model = Sequential()
    for i in range(depth):
        model.add(
            Conv1D(
                64,
                3,
                dilation_rate=2 if dilated else 1,
                activation="relu",
                input_shape=(X_train.shape[1], 1) if i == 0 else None
            )
        )
        model.add(MaxPooling1D(2))
    model.add(Flatten())
    model.add(Dense(128, activation="relu"))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(optimizer="adam", loss="binary_crossentropy")
    return model

for depth in [1, 2, 3, 4]:
    cnn = build_cnn(depth, dilated=False)
    cnn.fit(X_train_dl, y_train, epochs=10, batch_size=32, verbose=0)
    yp_prob = cnn.predict(X_test_dl).ravel()
    yp = (yp_prob > 0.5).astype(int)
    results.append([
        f"1D-CNN ({depth} layers)",
        f"{accuracy_score(y_test, yp)*100:.2f}",
        "—",
        f"{roc_auc_score(y_test, yp_prob)*100:.2f}",
        "—"
    ])

    dcnn = build_cnn(depth, dilated=True)
    dcnn.fit(X_train_dl, y_train, epochs=10, batch_size=32, verbose=0)
    yp_prob = dcnn.predict(X_test_dl).ravel()
    yp = (yp_prob > 0.5).astype(int)
    results.append([
        f"1D-DCNN ({depth} layers)",
        f"{accuracy_score(y_test, yp)*100:.2f}",
        "—",
        f"{roc_auc_score(y_test, yp_prob)*100:.2f}",
        "—"
    ])

# ============================================================
# FINAL TABLE
# ============================================================

df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy (%)", "95% CI", "AUC (%)", "95% CI"]
)

print(df.to_string(index=False))

/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step


/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step


/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step


/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step


/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


ValueError: Exception encountered when calling MaxPooling1D.call().

[1mNegative dimension size caused by subtracting 2 from 1 for '{{node sequential_21_1/max_pooling1d_36_1/MaxPool1d}} = MaxPool[T=DT_FLOAT, data_format="NHWC", explicit_paddings=[], ksize=[1, 1, 2, 1], padding="VALID", strides=[1, 1, 2, 1]](sequential_21_1/max_pooling1d_36_1/MaxPool1d/ExpandDims)' with input shapes: [?,1,1,64].[0m

Arguments received by MaxPooling1D.call():
  • inputs=tf.Tensor(shape=(None, 1, 64), dtype=float32)